# What This Walkthrough Does

This notebook uses synthetic data to show how to audit a platform research
dataset.

The goal is not to prove that one source is correct. The goal is to compare
what is visible through different access routes.

## Packages Used in This Notebook

This walkthrough uses:

- `pandas` to compare two synthetic datasets
- `Path` to save an audit report

The point is to learn audit logic: row counts, ID overlap, duplicates, missingness, and interpretation.


In [ ]:
# Path helps create output folders and report paths.
from pathlib import Path
# pandas is used for tables, ID comparison, duplicate checks, and missingness checks.
import pandas as pd


# Synthetic Reference and Observed Data

Imagine `reference` came from a small public-interface observation and
`observed` came from an API or platform research export.

In [ ]:
# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
reference = pd.DataFrame(
    [
        {"post_id": "a1", "url": "https://example.test/a1", "author": "u01", "created_at": "2026-01-01", "engagement": 10},
        {"post_id": "a2", "url": "https://example.test/a2", "author": "u02", "created_at": "2026-01-01", "engagement": 5},
        {"post_id": "a3", "url": "https://example.test/a3", "author": "u03", "created_at": "2026-01-02", "engagement": 22},
        {"post_id": "a4", "url": "https://example.test/a4", "author": "u04", "created_at": "2026-01-02", "engagement": 1},
    ]
)

observed = pd.DataFrame(
    [
        {"post_id": "a1", "url": "https://example.test/a1", "author": None, "created_at": "2026-01-01", "engagement": None},
        {"post_id": "a2", "url": "https://example.test/a2", "author": None, "created_at": "2026-01-01", "engagement": None},
        {"post_id": "a2", "url": "https://example.test/a2", "author": None, "created_at": "2026-01-01", "engagement": None},
        {"post_id": "x9", "url": "https://example.test/x9", "author": None, "created_at": "2026-01-03", "engagement": None},
    ]
)

print(reference)
print(observed)


# Compare IDs

In [ ]:
id_col = "post_id"

ref_ids = set(reference[id_col])
obs_ids = set(observed[id_col])

common = ref_ids & obs_ids
missing_from_observed = ref_ids - obs_ids
extra_in_observed = obs_ids - ref_ids

print("Common IDs:", common)
print("Missing from observed:", missing_from_observed)
print("Extra in observed:", extra_in_observed)


Row counts alone are not enough. Two datasets can have similar row counts but
different IDs.

# Check Duplicates

In [ ]:
print("Reference duplicate IDs:", reference[id_col].duplicated().sum())
print("Observed duplicate IDs:", observed[id_col].duplicated().sum())


Duplicates can come from pagination errors, overlapping queries, reposts,
platform-side bugs, or researcher-side merging mistakes.

# Audit Missing Metadata

In [ ]:
missing_counts = observed.isna().sum()
completeness = 1 - observed.isna().mean()

print("Observed missing counts:")
print(missing_counts)

print("\nObserved completeness:")
print(completeness)


The observed dataset contains some IDs but strips author and engagement
metadata. That may be acceptable for one research question and fatal for
another.

# Write an Audit Report

In [ ]:
report = {
    "reference_rows": len(reference),
    "observed_rows": len(observed),
    "reference_unique_ids": len(ref_ids),
    "observed_unique_ids": len(obs_ids),
    "common_ids": len(common),
    "missing_from_observed": sorted(missing_from_observed),
    "extra_in_observed": sorted(extra_in_observed),
    "observed_duplicate_ids": int(observed[id_col].duplicated().sum()),
    "observed_missing_by_column": missing_counts.astype(int).to_dict(),
    "observed_completeness": completeness.to_dict(),
    "interpretation": [
        "The observed data is incomplete relative to the benchmark.",
        "Metadata stripping affects author and engagement fields.",
        "The duplicate ID a2 should be investigated before analysis.",
    ],
}

outdir = Path("data")
reports_dir = outdir / "reports"
# mkdir(..., parents=True, exist_ok=True) creates the folder and any missing parent folders, without failing if it already exists.
reports_dir.mkdir(parents=True, exist_ok=True)

report_path = reports_dir / "notebook_data_quality_audit.json"
# to_json(...) saves structured data as JSON, which is useful for raw API responses and provenance records.
pd.Series(report).to_json(report_path, indent=2)

print(report_path)


# Exercise

Answer:

- Which IDs are missing from the observed data?
- Which IDs appear only in the observed data?
- Which columns have been stripped?
- Which research questions survive this missingness?
- What would you report in a methods section?

# Key Takeaway

Data quality is not only about file size or row counts. It is about coverage,
IDs, duplicates, missing fields, and the implications of the access route.